# Проект: 
### Модель рекомендации тарифов

# Project:
### Tariff Recommendation Model

В распоряжение даны данные о поведении клиентов оператора мобильной связи «Мегалайн», которые уже перешли на тарифы: «Смарт» или «Ультра».

#### Цель проекта:
Построить модель для задачи классификации, способную проанализировать поведение клиентов и предложить пользователям новый тариф: «Смарт» или «Ультра» с точностью более 0.75

##### Ход исследования:
1. Загрузка данных:
   - Чтение файла с данным и сохранение его в датасет.
   - Изучение общей информацию.
2. Разделение исходных данных на выборки:
   - Обучающую.
   - Валидационную.
   - Тестовую .
3. Исследование качества разных моделей, в зависимости от разных гиперпараметры. Выводы исследования.
4. Проверка качества модели на тестовой выборке.
5. Проверка модели на вменяемость.
6. Общий вывод.

---

The dataset contains information about the behavior of clients of the mobile operator **Megaline** who have already switched to one of the two тариф plans: **Smart** or **Ultra**.

#### Project Objective:
To build a **classification model** capable of analyzing customer behavior and recommending a new tariff plan — **Smart** or **Ultra** — with an accuracy greater than **0.75**.

##### Research Plan:
1. **Data loading:**
   - Read the file and store it in a dataset.
   - Examine the general information.
2. **Splitting the original data into subsets:**
   - Training set
   - Validation set
   - Test set
3. **Evaluating the quality of different models** under various hyperparameter settings and summarizing the results.
4. **Testing the selected model** on the test set.
5. **Checking the model for sanity.**
6. **Overall conclusion.**

### Этап 1. Загрузка данных.

In [17]:
#Импортируем библиотеку Pandas
import pandas as pd

In [18]:
#Прочитаем файл и сохраним его в переменную
try:
    data = pd.read_csv('/Users/danielschollenberg/Desktop/users_behavior.csv')
except:
    data = pd.read_csv('https://code.s3.yandex.net//datasets/users_behavior.csv')

In [19]:
display(data.head())
print(data.info())

,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB
None


#### Вывод:
- Датасет состоит из 3214 объектов(строк) и 5 признаков(столбцов) следующего содержания:
   - alls — количество звонков.
   - minutes — суммарная длительность звонков в минутах.
   - messages — количество sms-сообщений.
   - mb_used — израсходованный интернет-трафик в Мб.
   - is_ultra — каким тарифом пользовался в течение месяца («Ультра» — 1, «Смарт» — 0).
- Целевым признаком является столбец is_ultra.
- Целевой признак является категориальным,  а задача модели классификационной.

### Этап 2. Разделение исходных данных на выборки.

In [20]:
#Импортируем модуль для деления на выборки
from sklearn.model_selection import train_test_split

In [21]:
#Разделение данных на обучающую и временную (включает в себя валидационную и тестовую) выборки
data_train, data_temp = train_test_split(data, 
                                         test_size=0.4, 
                                         random_state=12345)

#Дальнейшее разделение временной выборки на обучающую и валидационную выборки
data_test, data_valid = train_test_split(data_temp, 
                                         test_size=0.5, 
                                         random_state=12345)

In [22]:
#Разделим выборки по признакам

#Обучающая
train_features = data_train.drop(['is_ultra'], axis=1)
train_target = data_train['is_ultra']

#Тестовая
test_features = data_test.drop(['is_ultra'], axis=1)
test_target = data_test['is_ultra']

#Валидационная
valid_features = data_valid.drop(['is_ultra'], axis=1)
valid_target = data_valid['is_ultra']

#### Вывод:
- Мы поделили данные на три выборки обучающую, тестовую и валидационную.
- Так как спрятанной тестовой выборки нет, - все данные внутри одного датасета, - мы разбили данные на три части, в соотношении 3:1:1. 
- Далее каждую из выборок мы разбили по признакам.

### Этап 3. Исследование качества разных моделей.

In [23]:
#Импортируем модули моделей: дерево решений, случайный лес, логистическая регрессия
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier 
from sklearn.linear_model import LogisticRegression
#Импортируем модуль для оценки точности модели классификации
from sklearn.metrics import accuracy_score

In [24]:
#Проверим разную глубину на модели дерево решений
for depth in range(1, 10):
    model = DecisionTreeClassifier(max_depth=depth,random_state=12345) 
    model.fit(train_features, train_target)
    valid_predictions = model.predict(valid_features)
    print("max_depth =", depth, ": ", end='')
    print(accuracy_score(valid_target, valid_predictions)) 

max_depth = 1 : 0.7356143079315708
max_depth = 2 : 0.7744945567651633
max_depth = 3 : 0.7791601866251944
max_depth = 4 : 0.7744945567651633
max_depth = 5 : 0.7838258164852255
max_depth = 6 : 0.776049766718507
max_depth = 7 : 0.7993779160186625
max_depth = 8 : 0.7931570762052877
max_depth = 9 : 0.7807153965785381


#### Вывод: 
- Значение accurancy - доли правильных ответов, превышает 0.75 уже уже при глубине равной 2, однако далее высокого роста точности не происходит.

In [25]:
#Проверим разное количество деревьев на модели случайный лес
for trees_count in range(1, 10):
    model = RandomForestClassifier(random_state=12345, n_estimators=trees_count) 
    model.fit(train_features, train_target)
    valid_predictions = model.predict(valid_features)
    print("trees_count =", trees_count, ": ", end='')
    print(accuracy_score(valid_target, valid_predictions)) 

trees_count = 1 : 0.7402799377916018
trees_count = 2 : 0.7589424572317263
trees_count = 3 : 0.7573872472783826
trees_count = 4 : 0.7729393468118196
trees_count = 5 : 0.7667185069984448
trees_count = 6 : 0.7791601866251944
trees_count = 7 : 0.7807153965785381
trees_count = 8 : 0.7869362363919129
trees_count = 9 : 0.7838258164852255


#### Вывод:
- Значение accurancy превышает 0.75 при количестве деревьев равном 2, но далее особо роста точности нету.

In [26]:
#Проверим разное количество итераций на модели логистическая регрессия
for iter_count in [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000]:
    model = LogisticRegression(random_state=12345, solver='lbfgs', max_iter=iter_count)
    model.fit(train_features, train_target)
    valid_predictions = model.predict(valid_features)
    print("iter_count =", iter_count, ": ", end='')
    print(accuracy_score(valid_target, valid_predictions)) 

iter_count = 100 : 0.6842923794712286
iter_count = 200 : 0.6842923794712286
iter_count = 300 : 0.6842923794712286
iter_count = 400 : 0.6842923794712286
iter_count = 500 : 0.6842923794712286
iter_count = 600 : 0.6842923794712286
iter_count = 700 : 0.6842923794712286
iter_count = 800 : 0.6842923794712286
iter_count = 900 : 0.6842923794712286
iter_count = 1000 : 0.6842923794712286


#### Вывод:
- Значение accurancy не превышает 0.75 независимо от количества итераций.
- Так-же независимо от количества итераций доля правильных ответов остается одинаковой.

#### Итоговый вывод:
- Наиболее хорошо себя показали модели дерево решений и случайный лес.
- Наиболее оптимальным выбором модели для данной задачи является модель: дерево решений -
   - Дерево решений имеет долю правильных ответов 0.77 уже при глубине 2. 
   - Случайный лес превышает эту точность только при количестве деревьев более 5.
   - Хоть точность случайного леса и выше при количестве деревьев выше 7, чем точность дерева решений при глубине 9, однако в целом модель случайного леса куда более требовательна к вычислительным ресурсам, чем модель дерева решений, тем более при таком количестве деревьев. Модель же дерева решений с практически минимальной глубиной 2 - является наиболее оптимальным вариантом по соотношению точности и требовательности.
- Не будем тратить лишние ресурсы на вычислительные мощности и выберем модель дерево решений с глубиной 2.

### Этап 4. Проверка качества модели на тестовой выборке.

In [27]:
#Создадим модель дерево решений с глубиной 2 не локально
model = DecisionTreeClassifier(max_depth=2,random_state=12345)
#Обучим ее
model.fit(train_features, train_target)
#Получим ответы
test_predictions = model.predict(test_features)
#Доля правильных ответов
model_accuracy = accuracy_score(test_target, test_predictions)
#Вывыдем их на экран
print("Модель дерева решений с глубиной 2 имеет долю правильных ответов на тестовой выборки:")
print(model_accuracy) 

Модель дерева решений с глубиной 2 имеет долю правильных ответов на тестовой выборки:
0.7822706065318819


#### Вывод:
- При проверке модели дерево решений с глубиной 2 на тестовой выборке доля правильных ответов оказалась даже больше чем при проверки на валидационной выборки: 0,77 против 0,78.
- Закрепляем выбор модели дерево решений с глубиной 2 в качестве наиболее оптимальной и подходящей модели для данной задачи.

### Этап 5. Проверка модели на вменяемость.

- Проверка модели на вменяемость - это процесс сравнения ее производительности с некоторым базовым уровнем, который можно считать случайным или наивным предсказанием. Это помогает определить, действительно ли модель информативна и превосходит ли она простые предсказания.
- Доля правильных ответов выбранной нами модели относители модели со случайным предсказанием с вероятностью 50/50 сушественно выше: 0.78 против 0.5. 
- Остается проверить выбранную нами модель на вменяемость относительно наивной модели.

In [28]:
#Импортитуруем простую модель, использующуяся в качестве базовой 
#для оценки производительности других, более сложных моделей - 
#в нашем случае: дерева решений с глубиной 2
from sklearn.dummy import DummyClassifier

In [29]:
#Создание базовой модели (наивная модель)
dummy_model = DummyClassifier(strategy="most_frequent")
#Обучение базовой модели
dummy_model.fit(train_features, train_target)
#Предсказание с использованием базовой модели
dummy_predictions = dummy_model.predict(test_features)
#Доля правильных ответов
dummy_accuracy = accuracy_score(test_target, dummy_predictions)

In [30]:
#Сравним производительность выбранной нами модели с базовой моделью и вывыдем показатели
if model_accuracy > dummy_accuracy:
    print("Модель дерева решений с глубиной 2 имеет долю правильных ответов выше, чем у базовой модели на:")
    #Посчитаем разницу долей в процентах и оставим только 2 знака после запятой
    print(round((model_accuracy/dummy_accuracy-1)*100, 2), '%')
    print()
    print("Доля правильных ответов модели дерева решений с глубиной 2:")
    print(model_accuracy) 
    print()
    print("Доля правильных ответов базовой модели:")
    print(dummy_accuracy)
else:
    print("Модель дерева решений с глубиной 2 имеет долю правильных ответов ниже, чем у базовой модели на:")
    print(round((model_accuracy/dummy_accuracy-1)*100, 2), '%')
    print()
    print("Доля правильных ответов модели дерева решений с глубиной 2:")
    print(model_accuracy) 
    print()
    print("Доля правильных ответов базовой модели:")
    print(dummy_accuracy)

Модель дерева решений с глубиной 2 имеет долю правильных ответов выше, чем у базовой модели на:
10.79 %

Доля правильных ответов модели дерева решений с глубиной 2:
0.7822706065318819

Доля правильных ответов базовой модели:
0.7060653188180405


#### Вывод:
- Выбранная нами модель: модель дерева решений с глубиной 2, оказалась более чем на 10% точнее чем базовая модель. Хотя стоит отметить, что базовая модель тоже справилась неплохо - доля правильных ответов более 0.7.
- Выбранная модель прошла проверку на вменяемость.

### Этап 6. Общий вывод:

### Stage 6. Overall Conclusion:

- Было проведено исследование оценки наиболее оптимальной модели для задачи классификации рекомендации тарифов «Смарт» и «Ультра» для пользователей оператора связи «Мегалайн».
- Для оценки были рассмотрены три модели:
   - дерево решений  
   - случайный лес  
   - логистическая регрессия  
- Наиболее оптимальной в итоге оказалась модель:
   - дерево решений с глубиной 2  
- Однако, если для бизнеса не критичны вычислительные затраты, можно выбрать модель случайного леса с количеством деревьев 9 и более — она показывает немного более высокую точность. Модель логистической регрессии показала наихудшие результаты и не была выбрана.
- Также в конце исследования была проведена проверка модели на вменяемость для выбранной модели (дерево решений с глубиной 2). В результате модель успешно прошла проверку.

---

- A study was conducted to identify the most suitable model for the classification task of recommending **Smart** and **Ultra** tariff plans for users of the mobile operator **Megaline**.
- Three models were evaluated:
   - Decision Tree  
   - Random Forest  
   - Logistic Regression  
- The most optimal model turned out to be:
   - a **Decision Tree with a depth of 2**  
- However, if computational resources are not a limiting factor, a **Random Forest model with 9 or more trees** can be considered, as it provides slightly higher accuracy. The Logistic Regression model performed worse and was not selected.
- Additionally, a sanity check was performed on the selected model (Decision Tree with depth 2), and the model successfully passed this validation.